In [ ]:
# ============================================================
# KONFIGURASI MODEL  --  jalankan cell ini PERTAMA
# Semua path output dibuat dari MODEL_TAG, sehingga model
# t5-large TIDAK menimpa file t5-small (asl_t5_finetuned, dst).
# ============================================================
MODEL_NAME = "google-t5/t5-large"
MODEL_TAG  = "t5_large"          # ganti ke nama lain untuk eksperimen baru

PREFIX = "translate ASL gloss to English: "
MAX_INPUT_LENGTH  = 128
MAX_TARGET_LENGTH = 128

# --- Hyperparameter (disetel untuk t5-large) ---
LEARNING_RATE    = 3e-4
NUM_EPOCHS       = 3
TRAIN_BATCH_SIZE = 4             # batch kecil + akumulasi gradien agar muat di GPU
EVAL_BATCH_SIZE  = 8
GRAD_ACCUM_STEPS = 4             # batch efektif = 4 x 4 = 16 (sama seperti t5-small)
OPTIMIZER        = "adafactor"   # hemat memori dibanding adamw, standar untuk T5
USE_GRADIENT_CHECKPOINTING = True  # matikan jika di A100 dan memori masih longgar

# --- Path lokal Colab ---
TRAINING_DIR    = f"/content/asl_{MODEL_TAG}_training"
FINAL_MODEL_DIR = f"/content/asl_{MODEL_TAG}_finetuned"
ONNX_DIR        = f"/content/asl_{MODEL_TAG}_onnx_opencv"
ONNX_ZIP_PATH   = f"/content/asl_{MODEL_TAG}_onnx_opencv.zip"
PREDICTIONS_CSV = f"/content/asl_{MODEL_TAG}_test_predictions.csv"

# --- Path Google Drive ---
DRIVE_ROOT       = "/content/drive/MyDrive/ASL"
DRIVE_MODEL_DIR  = f"{DRIVE_ROOT}/asl_{MODEL_TAG}_finetuned"
DRIVE_ONNX_DIR   = f"{DRIVE_ROOT}/asl_{MODEL_TAG}_onnx_opencv"

# Pengaman: jangan sampai path bentrok dengan model t5-small lama
_OLD_SMALL_PATHS = {
    "/content/asl_t5_training",
    "/content/asl_t5_finetuned",
    "/content/asl_t5_onnx_opencv",
    "/content/asl_t5_onnx_opencv.zip",
    "/content/asl_test_predictions.csv",
    f"{DRIVE_ROOT}/asl_t5_finetuned",
    f"{DRIVE_ROOT}/asl_t5_onnx_opencv",
}
_new_paths = [
    TRAINING_DIR, FINAL_MODEL_DIR, ONNX_DIR, ONNX_ZIP_PATH,
    PREDICTIONS_CSV, DRIVE_MODEL_DIR, DRIVE_ONNX_DIR,
]
_clash = [p for p in _new_paths if p in _OLD_SMALL_PATHS]
assert not _clash, f"Path bentrok dengan t5-small lama: {_clash}"

print("Model       :", MODEL_NAME)
print("Drive model :", DRIVE_MODEL_DIR)
print("Drive ONNX  :", DRIVE_ONNX_DIR)

In [ ]:
!pip install -q \
    "transformers==4.46.3" \
    "datasets==3.1.0" \
    "accelerate>=1.1.1" \
    "sentencepiece" \
    "sacrebleu" \
    "pandas"

!pip install -q "kagglehub[pandas-datasets]"

import os
import kagglehub
import pandas as pd

from kagglehub import KaggleDatasetAdapter

DATASET_HANDLE = (
    "thedevastator/"
    "unlock-the-power-of-english-asl-with-aslg-pc12-c"
)

# Unduh atau gunakan salinan dataset dalam cache KaggleHub
dataset_dir = kagglehub.dataset_download(DATASET_HANDLE)

print("Dataset directory:", dataset_dir)
print("\nFiles available:")

for root, dirs, files in os.walk(dataset_dir):
    for name in files:
        relative_path = os.path.relpath(
            os.path.join(root, name),
            dataset_dir
        )
        print(relative_path)

In [ ]:
# Ganti dengan nama file sebenarnya dari output Cell 1
file_path = "train.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    file_path,
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

In [ ]:
# Bersihkan teks tanpa mengubah isi kalimat
for col in ["gloss", "text"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

# Periksa kembali hasilnya
print("Shape:", df.shape)
print("Kolom:", df.columns.tolist())

display(df.head(10))

In [ ]:
# Sesuaikan nama kolom dengan kode training kita
if "sentence" not in df.columns and "text" in df.columns:
    df = df.rename(columns={"text": "sentence"})

print("Kolom:", df.columns.tolist())
print("Shape:", df.shape)

display(df[["gloss", "sentence"]].head(10))

In [ ]:
from sklearn.model_selection import train_test_split

# Pastikan kolom yang diperlukan tersedia
assert {"gloss", "sentence"}.issubset(df.columns), (
    f"Kolom tidak sesuai: {df.columns.tolist()}"
)

# Bersihkan karakter BOM dan spasi
for col in ["gloss", "sentence"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

# Hapus data kosong
df = df.dropna(subset=["gloss", "sentence"])
df = df[
    (df["gloss"] != "") &
    (df["sentence"] != "")
]

# Hapus pasangan gloss-kalimat yang identik
before = len(df)
df = df.drop_duplicates(
    subset=["gloss", "sentence"]
).reset_index(drop=True)

print("Duplikat dihapus:", before - len(df))
print("Total pasangan valid:", len(df))

# Bagi menjadi 80% training, 10% validation, 10% testing
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

print("\nJumlah data:")
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

# Periksa beberapa contoh
display(train_df.head(5))

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# MODEL_NAME dan PREFIX diambil dari cell konfigurasi di atas

# Muat tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ubah DataFrame Pandas menjadi Hugging Face Dataset
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
    "validation": Dataset.from_pandas(
        val_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
    "test": Dataset.from_pandas(
        test_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
})

print(raw_datasets)
print("Tokenizer berhasil dimuat:", MODEL_NAME)

In [ ]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):
    # Input model: gloss ASL
    inputs = [
        PREFIX + gloss
        for gloss in examples["gloss"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    # Target model: kalimat bahasa Inggris
    targets = tokenizer(
        text_target=examples["sentence"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = targets["input_ids"]

    return model_inputs


tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["gloss", "sentence"],
    desc="Tokenizing gloss-to-text dataset",
)

print(tokenized_datasets)

In [ ]:
sample = tokenized_datasets["train"][0]

print("Input IDs:")
print(sample["input_ids"])

print("\nDecoded input:")
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True,
    )
)

print("\nDecoded target:")
print(
    tokenizer.decode(
        sample["labels"],
        skip_special_tokens=True,
    )
)

print("\nJumlah data:")
for split in tokenized_datasets:
    print(split, ":", len(tokenized_datasets[split]))

In [ ]:
import os
import inspect
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

# Pastikan runtime Colab menggunakan GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU belum aktif. Pilih Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

# T5 tidak stabil di fp16, jadi gunakan bf16 jika GPU mendukung (A100/L4/H100)
USE_BF16 = torch.cuda.is_bf16_supported()
if not USE_BF16:
    print(
        "PERINGATAN: GPU ini tidak mendukung bf16 (mis. T4). "
        "Training akan berjalan di fp32 dan jauh lebih lambat; "
        "T5-large sangat disarankan di A100 atau L4."
    )

# Muat model pretrained
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Padding dilakukan secara dinamis dalam setiap batch
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=8,
)

# TRAINING_DIR diambil dari cell konfigurasi di atas

args_kwargs = dict(
    output_dir=TRAINING_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    optim=OPTIMIZER,
    warmup_ratio=0.03,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=100,
    bf16=USE_BF16,
    fp16=False,  # fp16 tidak stabil pada T5 (loss bisa NaN)
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    seed=42,
    dataloader_num_workers=2,
)

# Menyesuaikan nama argumen dengan versi Transformers
training_params = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

if "eval_strategy" in training_params:
    args_kwargs["eval_strategy"] = "epoch"
else:
    args_kwargs["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**args_kwargs)

# Menyesuaikan API Trainer dengan versi Transformers
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Model siap dilatih:", MODEL_NAME)
print("Output training :", TRAINING_DIR)
print(
    "Batch efektif    :",
    TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
)
print("Train:", len(tokenized_datasets["train"]))
print("Validation:", len(tokenized_datasets["validation"]))

In [ ]:
train_result = trainer.train()

print("Training selesai.")
print("Training loss:", train_result.training_loss)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import shutil

from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# FINAL_MODEL_DIR dan DRIVE_MODEL_DIR diambil dari cell konfigurasi di atas

# trainer memuat checkpoint terbaik berdasarkan validation loss
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Salin model ke Google Drive (folder terpisah dari model t5-small)
os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)

shutil.copytree(
    FINAL_MODEL_DIR,
    DRIVE_MODEL_DIR,
    dirs_exist_ok=True,
)

print("Model lokal :", FINAL_MODEL_DIR)
print("Model Drive :", DRIVE_MODEL_DIR)
print("Isi direktori:", os.listdir(FINAL_MODEL_DIR))

In [ ]:
import torch
import pandas as pd

from tqdm.auto import tqdm
from sacrebleu.metrics import BLEU, CHRF

model = trainer.model
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

test_data = raw_datasets["test"]

predictions = []
references = []
test_glosses = []

BATCH_SIZE = EVAL_BATCH_SIZE

for start in tqdm(
    range(0, len(test_data), BATCH_SIZE),
    desc="Evaluating test set",
):
    end = min(start + BATCH_SIZE, len(test_data))
    batch = test_data[start:end]

    prompts = [
        PREFIX + gloss
        for gloss in batch["gloss"]
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=2,
            do_sample=False,
        )

    batch_predictions = tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )

    predictions.extend(batch_predictions)
    references.extend(batch["sentence"])
    test_glosses.extend(batch["gloss"])

# Hitung metrik
bleu_metric = BLEU(lowercase=True)
chrf_metric = CHRF(word_order=2, lowercase=True)

bleu_score = bleu_metric.corpus_score(
    predictions,
    [references],
)

chrf_score = chrf_metric.corpus_score(
    predictions,
    [references],
)

print(f"Test BLEU: {bleu_score.score:.2f}")
print(f"Test chrF++: {chrf_score.score:.2f}")

# Periksa hasil terjemahan secara manual
results_df = pd.DataFrame({
    "gloss": test_glosses,
    "reference": references,
    "prediction": predictions,
})

display(results_df.head(20))

# Simpan seluruh prediksi agar bisa dianalisis
results_df.to_csv(
    PREDICTIONS_CSV,
    index=False,
)
print("Prediksi disimpan di:", PREDICTIONS_CSV)

In [ ]:
!pip install -q "optimum-onnx[onnxruntime]" onnx

In [ ]:
%pip uninstall -y diffusers

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

MODEL_DIR = DRIVE_MODEL_DIR

assert os.path.isdir(MODEL_DIR), (
    f"Checkpoint tidak ditemukan: {MODEL_DIR}"
)

print("Checkpoint tersedia:")
print(os.listdir(MODEL_DIR))

In [ ]:
%pip show optimum-onnx
%pip show transformers
%pip show huggingface-hub

In [ ]:
import os
import shutil

MODEL_DIR = DRIVE_MODEL_DIR
# ONNX_DIR diambil dari cell konfigurasi di atas

shutil.rmtree(ONNX_DIR, ignore_errors=True)

!optimum-cli export onnx \
    --model "{MODEL_DIR}" \
    --task text2text-generation \
    --opset 17 \
    "{ONNX_DIR}"

In [ ]:
import os

for root, dirs, files in os.walk(ONNX_DIR):
    for filename in files:
        print(os.path.join(root, filename))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
import onnx

ONNX_DIR = Path(ONNX_DIR)
DRIVE_ONNX_DIR = Path(DRIVE_ONNX_DIR)

assert (ONNX_DIR / "encoder_model.onnx").exists()
assert (ONNX_DIR / "decoder_model.onnx").exists()

# Simpan hasil ekspor ke Google Drive
shutil.copytree(
    ONNX_DIR,
    DRIVE_ONNX_DIR,
    dirs_exist_ok=True,
)

# Validasi kedua graph
for filename in ["encoder_model.onnx", "decoder_model.onnx"]:
    path = DRIVE_ONNX_DIR / filename
    onnx.checker.check_model(str(path))
    print(f"[VALID] {filename} — {path.stat().st_size / 1e6:.2f} MB")

print("\nFolder ONNX tersimpan:", DRIVE_ONNX_DIR)

In [ ]:
import torch
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM

MODEL_DIR = DRIVE_ONNX_DIR

tokenizer_onnx = AutoTokenizer.from_pretrained(MODEL_DIR)

model_onnx = ORTModelForSeq2SeqLM.from_pretrained(
    MODEL_DIR,
    provider="CPUExecutionProvider",
    use_cache=False,
)

gloss = "MEMBERSHIP PARLIAMENT SEE MINUTE"
prompt = PREFIX + gloss

inputs = tokenizer_onnx(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=128,
)

with torch.inference_mode():
    output_ids = model_onnx.generate(
        **inputs,
        max_new_tokens=48,
        num_beams=2,
        do_sample=False,
        use_cache=False,
    )

prediction = tokenizer_onnx.decode(
    output_ids[0],
    skip_special_tokens=True,
)

print("Input gloss :", gloss)
print("Prediksi    :", prediction)

In [ ]:
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer

MODEL_DIR = DRIVE_ONNX_DIR

tokenizer_onnx = AutoTokenizer.from_pretrained(MODEL_DIR)

model_onnx = ORTModelForSeq2SeqLM.from_pretrained(
    MODEL_DIR,
    provider="CPUExecutionProvider",
    use_cache=False,
    use_merged=False,
)

print("ONNX model loaded successfully.")

In [ ]:
import torch

for i in range(min(5, len(raw_datasets["test"]))):
    sample = raw_datasets["test"][i]

    inputs = tokenizer_onnx(
        PREFIX + sample["gloss"],
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    with torch.inference_mode():
        output_ids = model_onnx.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=2,
            do_sample=False,
            use_cache=False,
        )

    prediction = tokenizer_onnx.decode(
        output_ids[0],
        skip_special_tokens=True,
    )

    print(f"\nContoh {i + 1}")
    print("Gloss :", sample["gloss"])
    print("Target:", sample["sentence"])
    print("Prediksi:", prediction)

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

# Folder hasil ekspor ONNX
MODEL_DIR = Path(ONNX_DIR)

# Pastikan file utama tersedia
required_files = [
    "encoder_model.onnx",
    "decoder_model.onnx",
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "spiece.model",
]

missing = [
    name for name in required_files
    if not (MODEL_DIR / name).is_file()
]

if missing:
    raise FileNotFoundError(
        "File berikut tidak ditemukan:\n"
        + "\n".join(missing)
    )

# Buat satu ZIP yang berisi seluruh folder model
ZIP_PATH = Path(ONNX_ZIP_PATH)

with ZipFile(ZIP_PATH, "w", ZIP_DEFLATED) as zf:
    for path in MODEL_DIR.rglob("*"):
        if path.is_file():
            # Pertahankan struktur folder di dalam ZIP
            arcname = Path(MODEL_DIR.name) / path.relative_to(MODEL_DIR)
            zf.write(path, arcname)

print("ZIP berhasil dibuat!")
print(f"Jumlah file: {sum(1 for p in MODEL_DIR.rglob('*') if p.is_file())}")
print(f"Ukuran ZIP: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

# Mulai unduhan ke komputer melalui browser
files.download(str(ZIP_PATH))

In [ ]:
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM
import torch

onnx_tokenizer = AutoTokenizer.from_pretrained(str(ONNX_DIR))

onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
    str(ONNX_DIR),
    provider="CPUExecutionProvider",
)

sample = raw_datasets["test"][0]
gloss = sample["gloss"]

inputs = onnx_tokenizer(
    PREFIX + gloss,
    return_tensors="pt",
    truncation=True,
    max_length=128,
)

with torch.inference_mode():
    output_ids = onnx_model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=2,
        do_sample=False,
    )

prediction = onnx_tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True,
)

print("Input gloss :", gloss)
print("Target      :", sample["sentence"])
print("ONNX output :", prediction)

In [ ]:
shutil.copytree(
    ONNX_DIR,
    DRIVE_ONNX_DIR,
    dirs_exist_ok=True,
)

print("ONNX tersimpan di:", DRIVE_ONNX_DIR)